## [1.1.0] - 2026-06-13

### Changed

#### 核心框架重构：数据获取职责上移到基类

**Model 基类**
- 新增 `_get_variable_value(name, is_optional)` 私有方法，封装变量解析逻辑
- 新增 `prepare_calculation_context()` 方法，统一合并 required 与 optional 变量为单一字典
- 简化 `update_input_variable()` 逻辑，移除对 `get_name()` / `get_value()` 接口的兼容
- `evaluate()` 流程变更：
  - 旧：`check_variables()` → `_model_function(optional_variables, **kwargs)` → 合并结果
  - 新：`check_variables()` → `prepare_calculation_context()` → `_model_function(context)` → 合并结果

**Auditor 基类**
- 继承 Model 基类的 `prepare_calculation_context()` 方法
- `evaluate()` 流程变更：
  - 旧：`check_variables()` → `_model_function(optional_variables, **kwargs)` → 返回原状态
  - 新：`check_variables()` → `super().prepare_calculation_context()` → `_model_function(context)` → 返回原状态
- `output_names` 属性保持返回空列表（审计器不产生新变量）

**计算/验证函数签名变更**

| 组件 | 1.0.0 签名 | 1.1.0 签名 |
|:---|:---|:---|
| Model 计算函数 | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor 验证函数 | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

**受影响的模型（22 个全部更新）**

所有模型的计算函数已按新签名重构，移除内部的 `kwargs.get(key, default)` 样板代码：

- 广告漏斗（2 个）
- 成本模块（3 个）
- 交易模块（4 个）
- 费用模块（2 个）
- 收入与利润（4 个）
- 财务指标（5 个）
- 占位模型（2 个）

**受影响的审计器（1 个）**

- `PriceArchitectureAuditor` 验证函数已按新签名重构

### Removed

- `Model.update_input_variable()` 中对 `get_name()` / `get_value()` 接口的支持（统一使用 `name` + `expected_value` 模式）

### 设计收益

| 收益 | 说明 |
|:---|:---|
| **消除样板代码** | 计算/验证函数不再需要编写 `kwargs.get(key, default)` 重复逻辑 |
| **单一职责** | 基类负责数据获取，子类负责业务计算/验证 |
| **更易测试** | 计算/验证函数只依赖一个字典参数，可独立单元测试 |
| **接口更清晰** | 函数签名从 2 个参数简化为 1 个参数 |

### Migration Guide（从 1.0.0 升级到 1.1.0）

**对于自定义 Model 子类：**

1. 更新计算函数签名：
   ```python
   # 1.0.0
   def calculate_xxx(optional_variables: dict, **kwargs) -> dict:
       value = kwargs.get(key, optional_variables[key])
   
   # 1.1.0
   def calculate_xxx(variables: dict) -> dict:
       value = variables[key]
   ```

---

## [1.0.0] - 2026-06-12

### Added

#### 核心框架
- 新增 `Variable` 基类，支持 min/exp/max 边界定义及随机采样
- 新增 `Model` 基类，支持必需/可选变量验证与链式执行
- 新增 `Auditor` 基类（Model 特化），用于验证跨模型数据一致性
- 新增变量注册表（`variables/`），包含 20+ 业务变量定义

#### 模型库（22 个模型）

**广告漏斗**
- `AdvertisingEfficiencyGoogleSearchModel` — 广告预算 → 线索量
- `CostPerLeadGoogleSearchModel` — 每线索成本计算

**成本模块**
- `CostOfGoodsSoldModel` — 销货成本计算
- `ShippingCostModel` — 运费计算（基于零售价百分比）
- `TotalCostModel` — 总运营成本聚合（不含启动成本）

**交易模块**
- `DeductionRateModel` — 扣率聚合（运费+关税+渠道加价）
- `OrderModel` — 线索 → 订单转化
- `UnitFobModel` — 零售价 → FOB 价（逆向定价瀑布）
- `UnitContributionMarginModel` — 单位贡献毛利

**费用模块**
- `MonthlyExpenseModel` — 月度费用聚合
- `TotalExpenseModel` — 期间费用扩展

**收入与利润**
- `RevenueModel` — 收入计算（基于 FOB 价）
- `ProfitModel` — 运营毛利
- `NetIncomeModel` — 税后净利润
- `FreeCashFlowModel` — 自由现金流

**财务指标**
- `CacModel` — 客户获取成本
- `RoasModel` — 广告支出回报率
- `RoiModel` — 投资回报率（基于启动成本）
- `MarketPriceModel` — 公司估值（PE 倍数法）
- `PriceArchitectureModel` — 价格分解（单位层面）

**占位模型**
- `CapitalExpenditureModel` — 资本支出（当前返回 0）
- `DepreciationModel` — 折旧（当前返回 0）

#### 审计器
- `PriceArchitectureAuditor` — 验证零售价 = COGS + 运费 + 关税 + 渠道毛利 + 净利润

#### 分析模块（6 个）
- `break_even_analysis` — 多变量盈亏平衡分析
- `comparative_statics` — 三点敏感性分析与弹性计算
- `stochastic_contribution_analysis` — 蒙特卡洛贡献度分析
- `run_monte_carlo` — 蒙特卡洛模拟
- `stochastic_bivariate_simulation` — 双变量回归分析
- `run_two_way_sensitivity_analysis` — 双变量网格敏感性分析

#### 可视化（7 个视图）
- `break_even_view` — 盈亏平衡结果表格
- `comparative_statics_view` — 敏感性分析表格
- `contribution_pie_view` — 贡献度饼图
- `histogram_distribution_view` — 蒙特卡洛直方图
- `linear_regression_view` — 回归散点图 + 趋势线
- `two_way_sensitivity_heatmap_view` — 热力图

#### 管道与执行引擎
- `PipelineComposer` — 场景化管道构建（支持预定义场景、动态追加、场景合并）
- `runner.py` — 4 种执行模式（链式执行、基准场景、随机迭代、变量扫描）
- 拓扑顺序验证 — 防止变量覆盖冲突（DAG 性质）

#### 配置系统
- `settings.py` — 系统参数（采样步数、精度、审计容差、默认常量）
- `variable_names.py` — 50+ 变量名常量
- `messages.py` — 日志与错误信息模板
- `pipelines.py` — 预定义场景配置
- `formatting.py` — 变量格式化映射（货币、百分比、小数位数）

#### 工具函数
- `validation.py` — 变量缺失检测、管道拓扑验证
- `formatting.py` — 数值格式化（`fmt`）
- `logger.py` — 彩色控制台日志 + 可选文件输出

### Changed

#### 设计调整
- `TotalCostModel` 移除 `SetupCost`，仅聚合运营成本（COGS + 广告 + 运费）
- `SetupCost` 重新定位为投资项，用于 ROI 分母
- 统一 `_model_function` 签名为 `(optional_variables, **kwargs)`

#### 命名规范
- 变量常量使用 `SCREAMING_SNAKE_CASE`
- 字典 key 使用首字母大写的驼峰命名（如 `"Revenue"`, `"Cost"`）

### Fixed

- `monthly_expense` 映射错误 → 修正为 `MonthlyExpenseModel`
- `total_expense` 映射正确 → `TotalExpenseModel`
- `unit_fob` 模型缺失注册 → 已添加
- `price_architecture` 模型缺失注册 → 已添加
- `FreeCashFlowModel` 双层 fallback → 简化为单层
- `contribution_analysis` 缺少指标验证 → 已添加
- `regression_analysis` 方差零判断 → 改用容差
- `two_way_sensitivity` 过时注释 → 已移除

### Known Limitations（当前版本约束）

- 仅支持单一商品模型
- 订单交付时间滞后忽略
- 复购与客户生命周期价值忽略
- 转化率假设恒定
- 折旧与资本支出为占位（返回 0）
- 广告渠道归因假设 100% 贡献

### Future Roadmap

- 升级包模块（upgrade_cost, upgrade_price, upgrade_rate）
- 多渠道归因（区分广告渠道对订单的贡献）
- 批量折扣（COGS 非线性）
- 负利润税务处理（tax shield）
- 多商品支持（product_id 维度）
- 订单交付时间滞后
- 复购与 LTV


---



---

## [1.2.0] - 2026-06-16

### Added

#### Core Framework Auditors

* Added `UnitGrossProfitAuditor` — Validates `UnitGrossProfit = UnitFOBPrice - UnitEXWPrice`
* Added `UnitOperatingIncomeAuditor` — Validates `UnitOperatingIncome` waterfall (FOB - EXW - Marketing - FixedOverhead) while enforcing `UnitFreightExpense = 0` as a business rule
* Added `DeductionAuditor` — Validates deduction rate bounds (`0 < Rate < 1`) and reconciles the USD-denominated pricing waterfall

#### Refactored Models

* `NetIncomeModel` — Refactored to derive after-tax profitability directly from `OperatingIncome`
* `TotalExpenseModel` — Refactored to treat `ManagementExpense` and `SellingExpense` as optional inputs (defaulting to 0.0)
* `AdvertisingExpenseModel` — Added model for 1:1 marketing-to-advertising budget allocation

### Changed

#### Model Architecture Refinement

* **Parameter Flexibility:** Migrated several models (e.g., `TotalExpenseModel`) from mandatory required inputs to optional inputs, providing greater resilience for partial financial datasets.
* **Business Logic Enforcement:** Integrated "Forced Zero" logic into the `UnitOperatingIncomeAuditor` to ensure freight costs are correctly excluded from Brand-side profitability calculations regardless of raw input values.
* **Standardized Docstrings:** Updated all models and auditors with consistent reconciliation formulas, logic descriptions, and input/output mappings to match the `1.1.0` architectural standards.

#### Test Suite Updates

* **Engine Runner:** Updated `runner.py` test suite to reflect the new pipeline structure and updated mathematical traces (Units -> COGS -> Advertising -> Selling -> Total Expense).
* **Validation:** Added full unit test coverage for the three new auditors and refactored models, ensuring circuit breakers trigger correctly on reconciliation failures.

### Design Benefits

| Benefit | Description |
| --- | --- |
| **Pipeline Reliability** | New auditor suite ensures the Price Waterfall remains internally consistent during multi-stage execution |
| **Business Rule Compliance** | Centralized freight exclusion logic in the auditor prevents "leaky" cost accounting |
| **Input Robustness** | Optionality in cost models allows for lean execution paths without requiring dummy zero-value inputs |
| **Auditability** | Formulas and logic are now explicitly documented in class docstrings, facilitating easier peer review |


## [1.1.0] - 2026-06-13

### Changed

#### Core Framework Refactoring: Data Retrieval Responsibility Moved to Base Classes

**Model Base Class**
- Added `_get_variable_value(name, is_optional)` private method to encapsulate variable resolution logic
- Added `prepare_calculation_context()` method to uniformly merge required and optional variables into a single dictionary
- Simplified `update_input_variable()` logic, removed compatibility with `get_name()` / `get_value()` interfaces
- `evaluate()` flow changes:
  - Old: `check_variables()` → `_model_function(optional_variables, **kwargs)` → merge results
  - New: `check_variables()` → `prepare_calculation_context()` → `_model_function(context)` → merge results

**Auditor Base Class**
- Inherits `prepare_calculation_context()` method from Model base class
- `evaluate()` flow changes:
  - Old: `check_variables()` → `_model_function(optional_variables, **kwargs)` → return original state
  - New: `check_variables()` → `super().prepare_calculation_context()` → `_model_function(context)` → return original state
- `output_names` property remains returning empty list (auditors produce no new variables)

**Calculation/Validation Function Signature Changes**

| Component | 1.0.0 Signature | 1.1.0 Signature |
|:---|:---|:---|
| Model calculation function | `def func(optional_variables: dict, **kwargs) -> dict` | `def func(variables: dict) -> dict` |
| Auditor validation function | `def check(optional_variables: dict, **kwargs) -> None` | `def check(variables: dict) -> None` |

**Affected Models (22 total, all updated)**

All model calculation functions have been refactored to the new signature, removing internal `kwargs.get(key, default)` boilerplate:

- Advertising funnel (2)
- Cost modules (3)
- Deal modules (4)
- Expense modules (2)
- Revenue & Profit (4)
- Financial metrics (5)
- Placeholder models (2)

**Affected Auditors (1)**

- `PriceArchitectureAuditor` validation function refactored to new signature

### Removed

- Support for `get_name()` / `get_value()` interfaces in `Model.update_input_variable()` (unified to `name` + `expected_value` pattern)

### Design Benefits

| Benefit | Description |
|:---|:---|
| **Eliminate boilerplate** | Calculation/validation functions no longer need to write repetitive `kwargs.get(key, default)` logic |
| **Single responsibility** | Base class handles data retrieval; subclasses handle business calculation/validation |
| **Easier testing** | Calculation/validation functions depend on a single dictionary parameter, enabling independent unit testing |
| **Cleaner interface** | Function signature simplified from 2 parameters to 1 |

### Migration Guide (Upgrading from 1.0.0 to 1.1.0)

**For custom Model subclasses:**

1. Update calculation function signature:
   ```python
   # 1.0.0
   def calculate_xxx(optional_variables: dict, **kwargs) -> dict:
       value = kwargs.get(key, optional_variables[key])
   
   # 1.1.0
   def calculate_xxx(variables: dict) -> dict:
       value = variables[key]
   ```

---

## [1.0.0] - 2026-06-12

### Added

#### Core Framework
- Added `Variable` base class supporting min/exp/max boundaries and random sampling
- Added `Model` base class supporting required/optional variable validation and chained execution
- Added `Auditor` base class (Model specialization) for validating cross-model data consistency
- Added variable registry (`variables/`) containing 20+ business variable definitions

#### Model Library (22 models)

**Advertising Funnel**
- `AdvertisingEfficiencyGoogleSearchModel` — Ad budget → Leads
- `CostPerLeadGoogleSearchModel` — Cost per lead calculation

**Cost Modules**
- `CostOfGoodsSoldModel` — COGS calculation
- `ShippingCostModel` — Shipping cost calculation (percentage of retail price)
- `TotalCostModel` — Total operating cost aggregation (excluding setup cost)

**Deal Modules**
- `DeductionRateModel` — Deduction rate aggregation (shipping + tariff + channel markup)
- `OrderModel` — Leads → Orders conversion
- `UnitFobModel` — Retail price → FOB price (reverse pricing waterfall)
- `UnitContributionMarginModel` — Unit contribution margin

**Expense Modules**
- `MonthlyExpenseModel` — Monthly expense aggregation
- `TotalExpenseModel` — Period expense expansion

**Revenue & Profit**
- `RevenueModel` — Revenue calculation (based on FOB price)
- `ProfitModel` — Operating profit
- `NetIncomeModel` — After-tax net income
- `FreeCashFlowModel` — Free cash flow

**Financial Metrics**
- `CacModel` — Customer acquisition cost
- `RoasModel` — Return on advertising spend
- `RoiModel` — Return on investment (based on setup cost)
- `MarketPriceModel` — Company valuation (P/E multiple method)
- `PriceArchitectureModel` — Price decomposition (per-unit level)

**Placeholder Models**
- `CapitalExpenditureModel` — Capital expenditure (currently returns 0)
- `DepreciationModel` — Depreciation (currently returns 0)

#### Auditors
- `PriceArchitectureAuditor` — Validates retail price = COGS + shipping + tariff + channel margin + net profit

#### Analysis Modules (6)
- `break_even_analysis` — Multi-variable break-even analysis
- `comparative_statics` — Three-point sensitivity analysis and elasticity calculation
- `stochastic_contribution_analysis` — Monte Carlo contribution analysis
- `run_monte_carlo` — Monte Carlo simulation
- `stochastic_bivariate_simulation` — Bivariate regression analysis
- `run_two_way_sensitivity_analysis` — Two-variable grid sensitivity analysis

#### Visualization (7 views)
- `break_even_view` — Break-even results table
- `comparative_statics_view` — Sensitivity analysis table
- `contribution_pie_view` — Contribution pie chart
- `histogram_distribution_view` — Monte Carlo histogram
- `linear_regression_view` — Regression scatter plot + trend line
- `two_way_sensitivity_heatmap_view` — Heatmap

#### Pipeline & Execution Engine
- `PipelineComposer` — Scenario-based pipeline construction (supports predefined scenarios, dynamic appending, scenario merging)
- `runner.py` — 4 execution modes (chained execution, baseline scenario, random iteration, variable sweep)
- Topological order validation — Prevents variable overwrite conflicts (DAG property)

#### Configuration System
- `settings.py` — System parameters (sampling steps, precision, audit tolerances, default constants)
- `variable_names.py` — 50+ variable name constants
- `messages.py` — Log and error message templates
- `pipelines.py` — Predefined scenario configurations
- `formatting.py` — Variable formatting mappings (currency, percentage, decimal places)

#### Utility Functions
- `validation.py` — Variable missing detection, pipeline topology validation
- `formatting.py` — Numeric formatting (`fmt`)
- `logger.py` — Colored console logging + optional file output

### Changed

#### Design Adjustments
- `TotalCostModel` removed `SetupCost`, now aggregates only operating costs (COGS + ads + shipping)
- `SetupCost` repositioned as investment item, used as ROI denominator
- Unified `_model_function` signature to `(optional_variables, **kwargs)`

#### Naming Conventions
- Variable constants use `SCREAMING_SNAKE_CASE`
- Dictionary keys use PascalCase (e.g., `"Revenue"`, `"Cost"`)

### Fixed

- `monthly_expense` mapping error → corrected to `MonthlyExpenseModel`
- `total_expense` mapping correct → `TotalExpenseModel`
- `unit_fob` model missing registration → added
- `price_architecture` model missing registration → added
- `FreeCashFlowModel` double fallback → simplified to single layer
- `contribution_analysis` missing metric validation → added
- `regression_analysis` zero variance check → replaced with tolerance
- `two_way_sensitivity` outdated comments → removed

### Known Limitations (Current Version Constraints)

- Single-product model only
- Order delivery time lag ignored
- Repeat purchase and customer lifetime value ignored
- Conversion rates assumed constant
- Depreciation and CapEx are placeholders (return 0)
- Ad channel attribution assumes 100% contribution

### Future Roadmap

- Upgrade package module (upgrade_cost, upgrade_price, upgrade_rate)
- Multi-channel attribution (distinguish ad channel contributions)
- Volume discounts (non-linear COGS)
- Negative profit tax treatment (tax shield)
- Multi-product support (product_id dimension)
- Order delivery time lag
- Repeat purchase and LTV